# 🎨 ComfyUI Colab — FLUX.1-schnell GGUF Q5_K_S (chất lượng cao nhất dưới 15GB)

Kéo theo: Impact Pack + YOLO (mặt/tay/chân) + SAM (mặt) + **ComfyUI-GGUF** — chạy trên Colab free (T4 16GB).

**Tổng dung lượng model ~12.3 GB (✅ dưới 15GB)**. Thời gian ~30-40s/ảnh 1024×1024.


In [ ]:
# ===== CELL 1: Mount Drive + cài ComfyUI =====
BO_QUA_DRIVE = False  # @param {type:"boolean"}
WAI_FILE_ID = ""  # @param {type:"string"}  # dán ID khác nếu cần, để trống sẽ dùng link bên dưới
WAI_ID_MAC_DINH = "1ExEKvWcb3Sav-j0Pn24of55SejUN1fNn"  # link share anyone

import os, time, sys, glob, shutil, subprocess
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

log('GPU:')
!nvidia-smi --query-gpu=name,memory.total --format=csv

USE_DRIVE = False
if not BO_QUA_DRIVE:
    log('Kết nối Drive — nếu đứng >60s: tick BO_QUA_DRIVE, Runtime → Interrupt, chạy lại.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        USE_DRIVE = True
        log('Drive OK')
    except Exception as e:
        log(f'Drive lỗi ({e}) → ổ tạm')
else:
    log('Bỏ qua Drive')

SHARE_URL = 'https://drive.google.com/drive/folders/16rLwN_FVAx-P_-2lhxWkSfyIgpgbKT40'
SHARE_ID  = '16rLwN_FVAx-P_-2lhxWkSfyIgpgbKT40'
SHARE_ID2 = '1KOCCIVQ36RwIG6_1Y8qzOMf6rcY3QV_9'

def run(cmd):
    log(f'$ {cmd}')
    subprocess.run(cmd, shell=True)

if USE_DRIVE:
    log('--- 1) MyDrive cấp 1 ---')
    run('ls -lh "/content/drive/MyDrive" | head -n 100')
    log('--- 2) .shortcut-targets-by-id ---')
    run('ls -lh "/content/drive/MyDrive/.shortcut-targets-by-id" 2>&1 | head -n 100')
    for sid in (SHARE_ID, SHARE_ID2):
        sc = f'/content/drive/MyDrive/.shortcut-targets-by-id/{sid}'
        if os.path.isdir(sc):
            log(f'--- 3) Nội dung shortcut {sid} ---')
            run(f'ls -lhR "{sc}" 2>&1 | head -n 200')
    log('--- 4) Tìm mọi *.safetensors >1GB trong Drive (20s) ---')
    run('find "/content/drive/MyDrive" -type f -name "*.safetensors" -size +1G 2>/dev/null | head -n 50 | xargs -I{} sh -c \'ls -lh "{}"\'')
    log('--- 5) Tìm file tên *WAI* ---')
    run('find "/content/drive" -type f -iname "*wai*.safetensors" 2>/dev/null | head -n 20 | xargs -I{} sh -c \'ls -lh "{}"\' 2>&1')

def is_model_root(root):
    try:
        return os.path.isdir(os.path.join(root, 'checkpoints')) and (os.path.isdir(os.path.join(root, 'sams')) or os.path.isdir(os.path.join(root, 'ultralytics')))
    except:
        return False

ROOT = None
candidates = []
if USE_DRIVE:
    candidates.append(f'/content/drive/MyDrive/.shortcut-targets-by-id/{SHARE_ID}')
    candidates.append(f'/content/drive/MyDrive/.shortcut-targets-by-id/{SHARE_ID2}')
    for name in ('AI_Models','AI_models'):
        candidates.append(f'/content/drive/MyDrive/{name}')
    try:
        for n in os.listdir('/content/drive/MyDrive'):
            p = os.path.join('/content/drive/MyDrive', n)
            if os.path.isdir(p):
                candidates.append(p)
    except:
        pass
    for c in candidates:
        if is_model_root(c):
            ROOT = c; break
    if not ROOT:
        for c in candidates:
            if os.path.isdir(os.path.join(c, 'checkpoints')):
                ROOT = c; break

if not ROOT:
    ROOT = '/content/drive/MyDrive/AI_Models' if USE_DRIVE else '/content/AI_Models'
    log(f'⚠️ Không tự tìm thấy AI_Models, tạm dùng {ROOT}')

log(f'ROOT chọn: {ROOT}')
for pth in [f'{ROOT}/checkpoints', f'{ROOT}/vae', f'{ROOT}/clip', f'{ROOT}/gguf',
            f'{ROOT}/ultralytics/bbox', f'{ROOT}/sams']:
    os.makedirs(pth, exist_ok=True)

CKPT_DIR = f'{ROOT}/checkpoints'
VAE_DIR  = f'{ROOT}/vae'
CLIP_DIR = f'{ROOT}/clip'
GGUF_DIR = f'{ROOT}/gguf'
YOLO_DIR = f'{ROOT}/ultralytics/bbox'
SAM_DIR  = f'{ROOT}/sams'
open('/content/ckpt_dir.txt','w').write(CKPT_DIR)
open('/content/yolo_dir.txt','w').write(YOLO_DIR)
open('/content/sam_dir.txt','w').write(SAM_DIR)
open('/content/root_dir.txt','w').write(ROOT)

def find_wai_file():
    search_roots = [CKPT_DIR, ROOT]
    search_roots += [f'/content/drive/MyDrive/.shortcut-targets-by-id/{SHARE_ID}', f'/content/drive/MyDrive/.shortcut-targets-by-id/{SHARE_ID2}']
    if USE_DRIVE:
        try:
            for n in os.listdir('/content/drive/MyDrive/.shortcut-targets-by-id'):
                search_roots.append(f'/content/drive/MyDrive/.shortcut-targets-by-id/{n}')
                search_roots.append(f'/content/drive/MyDrive/.shortcut-targets-by-id/{n}/checkpoints')
        except:
            pass
    for r in search_roots:
        for name in ('WAI-illustrious.safetensors','wai-illustrious.safetensors','WAI.safetensors'):
            fp = os.path.join(r, name)
            if os.path.isfile(fp):
                try:
                    if os.path.getsize(fp) > 2*1024**3:
                        return fp
                except: pass
    try:
        out = subprocess.check_output('find "/content/drive/MyDrive" -type f -name "WAI-illustrious.safetensors" -size +2G 2>/dev/null | head -n 1', shell=True, text=True).strip()
        if out and os.path.isfile(out):
            return out
    except:
        pass
    return None

real_wai = find_wai_file()
if real_wai:
    log(f'✅ WAI tìm thấy: {real_wai} ({os.path.getsize(real_wai)/1024**3:.2f}GB)')
    dest = os.path.join(CKPT_DIR, 'WAI-illustrious.safetensors')
    if os.path.abspath(real_wai) != os.path.abspath(dest):
        if not os.path.isfile(dest) or os.path.getsize(dest) < 2*1024**3:
            try:
                if os.path.exists(dest): os.remove(dest)
                os.symlink(real_wai, dest)
                log(f'  → symlink {dest}')
            except:
                try:
                    shutil.copyfile(real_wai, dest)
                    log(f'  → copy {dest}')
                except Exception as e:
                    log(f'  link lỗi: {e}')
else:
    log('❌ Chưa thấy WAI >2GB trong Drive mount.')
    log('   Nguyên nhân: file WAI trong folder share là "Lối tắt đến tệp" — Colab mount thường KHÔNG hiện file shortcut.')
    log('   FIX: Acc A chia sẻ file GỐC → Acc B "Thêm lối tắt vào Drive → MyDrive/AI_Models/checkpoints"')
    log('   HOẶC dán ID file vào ô WAI_FILE_ID ở đầu Cell 1 này, chạy lại.')
    fid = (WAI_FILE_ID.strip() or WAI_ID_MAC_DINH.strip())
    if fid:
        if 'id=' in fid: fid = fid.split('id=')[1].split('&')[0]
        if '/d/' in fid: fid = fid.split('/d/')[1].split('/')[0]
        log(f'  Thử tải WAI bằng gdown ID={fid}')
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        dest = os.path.join(CKPT_DIR, 'WAI-illustrious.safetensors')
        import gdown
        try:
            gdown.download(id=fid, output=dest, quiet=False)
            if os.path.isfile(dest):
                log(f'✅ Tải xong WAI qua ID: {os.path.getsize(dest)/1024**3:.2f}GB')
        except Exception as e:
            log(f'  gdown lỗi: {e}')

log(f'CKPT_DIR={CKPT_DIR}')
!ls -lh "{CKPT_DIR}" | head -n 30

os.chdir('/content')
if os.path.isdir('/content/ComfyUI') and os.path.isfile('/content/ComfyUI/main.py'):
    log('ComfyUI đã có')
else:
    log('Clone ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
os.chdir('/content/ComfyUI')
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /content/req_notorch.txt
!pip install -r /content/req_notorch.txt
import torch
assert torch.cuda.is_available(), '❌ Không GPU'
log(f'PyTorch {torch.__version__} | {torch.cuda.get_device_name(0)}')
log('✅ Xong Cell 1')


In [ ]:
# ===== CELL 1B: Impact Pack + YOLO/SAM + ComfyUI-GGUF (nhẹ, không đè OpenCV/Torch) =====
import os, time, sys, subprocess
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

ROOT = open('/content/root_dir.txt').read().strip()
CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
YOLO_DIR = open('/content/yolo_dir.txt').read().strip()
SAM_DIR  = open('/content/sam_dir.txt').read().strip()
VAE_DIR  = f'{ROOT}/vae'
CLIP_DIR = f'{ROOT}/clip'
GGUF_DIR = f'{ROOT}/gguf'
for d in (VAE_DIR, CLIP_DIR, GGUF_DIR):
    os.makedirs(d, exist_ok=True)

CN = '/content/ComfyUI/custom_nodes'
os.makedirs(CN, exist_ok=True)
os.chdir(CN)

def clone(url, folder):
    path = os.path.join(CN, folder)
    if os.path.isdir(path) and os.listdir(path):
        log(f'{folder} đã có — bỏ qua clone')
        return
    log(f'Clone {folder}...')
    r = subprocess.run(['git','clone','--progress','--depth','1', url, path])
    if r.returncode != 0:
        raise RuntimeError(f'Clone {folder} thất bại')

clone('https://github.com/ltdrdata/ComfyUI-Impact-Pack.git', 'ComfyUI-Impact-Pack')
clone('https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git', 'ComfyUI-Impact-Subpack')
clone('https://github.com/city96/ComfyUI-GGUF.git', 'ComfyUI-GGUF')  # load FLUX GGUF

log('pip nhẹ: piexif dill segment-anything + ultralytics --no-deps (không cài lại opencv/torch)')
!pip install -q piexif dill segment-anything
!pip install -q ultralytics --no-deps

log('Symlink các thư mục model')
# ComfyUI-GGUF dùng key "unet_gguf" (fallback diffusion_models/unet) và "clip_gguf" (fallback text_encoders/clip)
# → symlink cả models/unet và models/unet_gguf tới GGUF_DIR; models/clip và models/text_encoders tới CLIP_DIR
pairs = [
    ('/content/ComfyUI/models/checkpoints',    CKPT_DIR),
    ('/content/ComfyUI/models/vae',            VAE_DIR),
    ('/content/ComfyUI/models/clip',           CLIP_DIR),
    ('/content/ComfyUI/models/text_encoders',  CLIP_DIR),
    ('/content/ComfyUI/models/unet',           GGUF_DIR),
    ('/content/ComfyUI/models/unet_gguf',      GGUF_DIR),
    ('/content/ComfyUI/models/ultralytics',    f'{ROOT}/ultralytics'),
    ('/content/ComfyUI/models/sams',           SAM_DIR),
]
for path, dest in pairs:
    os.makedirs(dest, exist_ok=True)
    if os.path.islink(path) or os.path.exists(path):
        !rm -rf "{path}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    os.symlink(dest, path)
    log(f'  {path} → {dest}')

print()
!ls /content/ComfyUI/custom_nodes | grep -iE 'impact|gguf'
log('✅ Xong Cell 1B — chạy Cell 2')


In [ ]:
# @title ⬇️ CELL 2 — Tải FLUX.1-schnell GGUF (tổng ~12.5GB, chất lượng nhất dưới 15GB) + YOLO + SAM
BO_QUA_MODEL = False  # @param {type:"boolean"}
# ↑ Tick nếu bạn đã chạy cell này trước đó và model đã có sẵn trên Drive.

import os, sys, time, shutil, subprocess

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

def read_dir(txt, fallback):
    if os.path.isfile(txt):
        d = open(txt).read().strip()
        if d: return d
    log(f'⚠️ Thiếu {txt} — dùng {fallback} (chạy Cell 1 trước)')
    return fallback

CKPT_DIR = read_dir('/content/ckpt_dir.txt', '/content/AI_Models/checkpoints')
YOLO_DIR = read_dir('/content/yolo_dir.txt', '/content/AI_Models/ultralytics/bbox')
SAM_DIR  = read_dir('/content/sam_dir.txt',  '/content/AI_Models/sams')
ROOT     = read_dir('/content/root_dir.txt',  '/content/AI_Models')
VAE_DIR  = f'{ROOT}/vae'
CLIP_DIR = f'{ROOT}/clip'
GGUF_DIR = f'{ROOT}/gguf'
for d in (CKPT_DIR, YOLO_DIR, SAM_DIR, VAE_DIR, CLIP_DIR, GGUF_DIR):
    os.makedirs(d, exist_ok=True)
log(f'ROOT={ROOT}')
log(f'  VAE={VAE_DIR}  CLIP={CLIP_DIR}  GGUF={GGUF_DIR}  YOLO={YOLO_DIR}  SAM={SAM_DIR}')

open('/content/model_type.txt','w').write('FLUX.1 [schnell] GGUF Q5_K_S')

os.environ.setdefault('HF_HUB_DISABLE_XET','1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY','1')

def ok_file(p, m):
    return os.path.isfile(p) and os.path.getsize(p) >= m

def curl_get(url, path, minb, headers=None):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    tmp = path+'.part'
    cmd = ['curl','-L','--fail','--retry','5','--retry-delay','2','--retry-all-errors','-C','-',
           '--connect-timeout','30','--max-time','0','-A','Mozilla/5.0','-o',tmp]
    if headers:
        for h in headers:
            cmd += ['-H', h]
    cmd.append(url)
    r = subprocess.run(cmd)
    if r.returncode != 0 or not os.path.isfile(tmp): return False
    if os.path.getsize(tmp) < minb:
        try: os.remove(tmp)
        except: pass
        return False
    shutil.move(tmp,path); return True

def hf_get(repo, fn, path, minb, ep=None):
    try: from huggingface_hub import hf_hub_download
    except:
        subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub'])
        from huggingface_hub import hf_hub_download
    import warnings; warnings.filterwarnings('ignore',message='.*pickle.*')
    os.environ.setdefault('HF_HUB_DISABLE_XET','1')
    log(f'  HF {repo}/{fn}' + (f' @ {ep}' if ep else ''))
    try:
        kw = dict(repo_id=repo, filename=fn, resume_download=True)
        if ep: kw['endpoint'] = ep
        sf = hf_hub_download(**kw)
        if os.path.getsize(sf) < minb:
            log('  file quá nhỏ'); return False
        shutil.copy2(sf, path); return True
    except Exception as e:
        log(f'  HF lỗi: {str(e).split(chr(10))[0][:200]}')
        return False

def download_any(label, path, minb, sources, required=True):
    if ok_file(path, minb):
        log(f'✅ {label} đã có ({os.path.getsize(path)/1e6:.1f} MB)'); return True
    log(f'⬇️  {label}')
    last = None
    for s in sources:
        kind = s[0]
        try:
            if kind == 'curl':
                good = curl_get(s[1], path, minb, headers=s[2] if len(s) > 2 else None)
            elif kind == 'hf':
                ep = s[3] if len(s) > 3 else None
                good = hf_get(s[1], s[2], path, minb, ep=ep)
            else:
                continue
            if good:
                log(f'✅ {label} ({os.path.getsize(path)/1e6:.1f} MB)'); return True
        except Exception as e:
            last = e; log(f'  thử mirror lỗi ({e})')
    msg = f'❌ Không tải được {label}' + (f' ({last})' if last else '')
    if required: raise RuntimeError(msg)
    log(msg+' — bỏ qua'); return False

# ============ FLUX.1-schnell GGUF Q5_K_S ============
if not BO_QUA_MODEL:
    log('===== FLUX.1-schnell GGUF Q5_K_S (phiên bản tốt nhất, ~12GB) =====')

    # VAE ae.safetensors (Comfy-Org đã xóa khỏi repo schnell → dùng mirror camenduru public)
    download_any('FLUX VAE ae.safetensors (~335MB)', os.path.join(VAE_DIR,'ae.safetensors'), 250_000_000, [
        ('curl','https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.safetensors?download=true'),
        ('curl','https://hf-mirror.com/camenduru/FLUX.1-dev/resolve/main/ae.safetensors?download=true'),
        ('hf','camenduru/FLUX.1-dev','ae.safetensors'),
        ('hf','camenduru/FLUX.1-dev','ae.safetensors','https://hf-mirror.com'),
    ], required=True)

    # CLIP-L — ở repo comfyanonymous/flux_text_encoders (public, không gated)
    download_any('FLUX CLIP-L clip_l.safetensors (~246MB)', os.path.join(CLIP_DIR,'clip_l.safetensors'), 200_000_000, [
        ('curl','https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors?download=true'),
        ('curl','https://hf-mirror.com/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors?download=true'),
        ('hf','comfyanonymous/flux_text_encoders','clip_l.safetensors'),
        ('hf','comfyanonymous/flux_text_encoders','clip_l.safetensors','https://hf-mirror.com'),
    ], required=True)

    # UNET Q5_K_S (city96)
    download_any('FLUX UNET Q5_K_S (~8.3GB)', os.path.join(GGUF_DIR,'flux1-schnell-Q5_K_S.gguf'), 7_500_000_000, [
        ('curl','https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q5_K_S.gguf?download=true'),
        ('curl','https://hf-mirror.com/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q5_K_S.gguf?download=true'),
        ('hf','city96/FLUX.1-schnell-gguf','flux1-schnell-Q5_K_S.gguf'),
        ('hf','city96/FLUX.1-schnell-gguf','flux1-schnell-Q5_K_S.gguf','https://hf-mirror.com'),
    ], required=True)

    # T5-XXL Q4_K_M (city96)
    download_any('T5-XXL Q4_K_M (~2.9GB)', os.path.join(CLIP_DIR,'t5-v1_1-xxl-encoder-Q4_K_M.gguf'), 2_500_000_000, [
        ('curl','https://huggingface.co/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/t5-v1_1-xxl-encoder-Q4_K_M.gguf?download=true'),
        ('curl','https://hf-mirror.com/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/t5-v1_1-xxl-encoder-Q4_K_M.gguf?download=true'),
        ('hf','city96/t5-v1_1-xxl-encoder-gguf','t5-v1_1-xxl-encoder-Q4_K_M.gguf'),
        ('hf','city96/t5-v1_1-xxl-encoder-gguf','t5-v1_1-xxl-encoder-Q4_K_M.gguf','https://hf-mirror.com'),
    ], required=True)

# ============ YOLO (mặt/tay/chân) ============
download_any('YOLO mặt face_yolov8m.pt (~52MB)', os.path.join(YOLO_DIR,'face_yolov8m.pt'), 40_000_000, [
    ('hf','Bingsu/adetailer','face_yolov8m.pt'),
    ('hf','Bingsu/adetailer','face_yolov8m.pt','https://hf-mirror.com'),
    ('curl','https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt?download=true'),
    ('curl','https://hf-mirror.com/Bingsu/adetailer/resolve/main/face_yolov8m.pt?download=true'),
], required=True)
download_any('YOLO tay hand_yolov8s.pt (~22MB)', os.path.join(YOLO_DIR,'hand_yolov8s.pt'), 15_000_000, [
    ('hf','Bingsu/adetailer','hand_yolov8s.pt'),
    ('hf','Bingsu/adetailer','hand_yolov8s.pt','https://hf-mirror.com'),
    ('curl','https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt?download=true'),
    ('curl','https://hf-mirror.com/Bingsu/adetailer/resolve/main/hand_yolov8s.pt?download=true'),
], required=True)
# Chú ý: Bingsu/adetailer KHÔNG có foot model. Dùng Claquasse/foot_anime_yolo (YOLO11m chuyên chân anime, ~40MB)
download_any('YOLO chân foot_anime_yolo11m_v3.pt (~40MB)', os.path.join(YOLO_DIR,'foot_anime_yolo11m_v3.pt'), 30_000_000, [
    ('curl','https://huggingface.co/Claquasse/foot_anime_yolo/resolve/main/foot_anime_yolo11m_v3.pt?download=true'),
    ('curl','https://hf-mirror.com/Claquasse/foot_anime_yolo/resolve/main/foot_anime_yolo11m_v3.pt?download=true'),
    ('hf','Claquasse/foot_anime_yolo','foot_anime_yolo11m_v3.pt'),
    ('hf','Claquasse/foot_anime_yolo','foot_anime_yolo11m_v3.pt','https://hf-mirror.com'),
], required=True)

# ============ SAM (chỉ cho mặt để tiết kiệm VRAM) ============
download_any('SAM ViT-B (chỉnh mặt, ~375MB)', os.path.join(SAM_DIR,'sam_vit_b_01ec64.pth'), 300_000_000, [
    ('curl','https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'),
    ('hf','segments-ai/sam_vit_b','sam_vit_b_01ec64.pth'),
    ('hf','segments-ai/sam_vit_b','sam_vit_b_01ec64.pth','https://hf-mirror.com'),
], required=True)

# ============ Kiểm tra tổng dung lượng ============
print()
log('===== 📊 Tổng dung lượng model trên Drive =====')
total = 0
for d in (VAE_DIR, CLIP_DIR, GGUF_DIR, YOLO_DIR, SAM_DIR):
    s = sum(os.path.getsize(os.path.join(d,f)) for f in os.listdir(d) if os.path.isfile(os.path.join(d,f)))
    total += s
    log(f'  {d}: {s/1e9:.2f} GB')
log(f'🟢 TỔNG: {total/1e9:.2f} GB / 15 GB  ({(total/15e9)*100:.1f}%)')
open('/content/total_size.txt','w').write(str(total))
log('✅ Xong Cell 2 — chạy Cell 3 (khởi chạy ComfyUI)')


In [ ]:
# @title ⚙️ CELL 3 — Khởi chạy ComfyUI
LISTEN = "0.0.0.0"  # @param {type:"string"}
PORT = 8188  # @param {type:"integer"}
TUNNEL = "cloudflared http2 (mạng VN)"  # @param ["cloudflared http2 (mạng VN)", "cloudflared quic", "không tunnel (chỉ Colab local)", "bỏ qua tunnel — ComfyUI only"]
CORS = True  # @param {type:"boolean"}
CORS_ORIGIN = "*"  # @param {type:"string"}

VRAM = "lowvram"  # @param ["lowvram", "Mặc định (T4)", "highvram", "normalvram", "novram", "cpu"]
RESERVE_VRAM_GB = 0.6  # @param {type:"slider", min:0.0, max:4.0, step:0.1}
CUDA_DEVICE = 0  # @param {type:"integer"}
CUDA_MALLOC = True  # @param {type:"boolean"}
DISABLE_SMART_MEMORY = False  # @param {type:"boolean"}

FORCE_PREC = "fp16"  # @param ["fp16", "fp32", "không ép"]
UNET_PREC = "Mặc định"  # @param ["Mặc định", "fp16-unet", "bf16-unet", "fp8_e4m3fn-unet"]
VAE_PREC = "fp16-vae"  # @param ["Mặc định", "fp16-vae", "fp32-vae", "bf16-vae", "cpu-vae"]

ATTENTION = "pytorch"  # @param ["pytorch", "split", "quad", "sage", "flash", "xformers (nếu có)", "Mặc định"]
PREVIEW = "auto"  # @param ["auto", "latent2rgb", "taesd", "none"]
FAST = False  # @param {type:"boolean"}
DISABLE_XFORMERS = True  # @param {type:"boolean"}

OUTPUT_DIR = "/content/ComfyUI/output"  # @param {type:"string"}
INPUT_DIR = "/content/ComfyUI/input"  # @param {type:"string"}
TEMP_DIR = "/content/ComfyUI/temp"  # @param {type:"string"}

KILL_OLD = True  # @param {type:"boolean"}
CHECK_MODELS = True  # @param {type:"boolean"}
WAIT_SECONDS = 180  # @param {type:"integer"}
EXTRA_ARGS = ""  # @param {type:"string"}
VERBOSE_LOG = True  # @param {type:"boolean"}

import os, sys, time, socket, re, subprocess, shutil, urllib.request

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

COMFY = '/content/ComfyUI'
assert os.path.isfile(f'{COMFY}/main.py'), '❌ Chưa có ComfyUI — chạy Cell 1 trước'

if CHECK_MODELS:
    log('Kiểm tra FLUX GGUF / YOLO / SAM / Impact + ComfyUI-GGUF...')
    def ls(p):
        try: return os.listdir(p)
        except: return []
    ggu, clip, vae, yolo, sams = (f'{COMFY}/models/unet', f'{COMFY}/models/clip', f'{COMFY}/models/vae',
                                 f'{COMFY}/models/ultralytics/bbox', f'{COMFY}/models/sams')
    print('  Impact Pack :', os.path.isdir(f'{COMFY}/custom_nodes/ComfyUI-Impact-Pack'))
    print('  Impact Sub  :', os.path.isdir(f'{COMFY}/custom_nodes/ComfyUI-Impact-Subpack'))
    print('  ComfyUI-GGUF:', os.path.isdir(f'{COMFY}/custom_nodes/ComfyUI-GGUF'))
    print('  unet (gguf) :', [f for f in ls(ggu) if f.endswith('.gguf')])
    print('  clip        :', ls(clip))
    print('  vae         :', ls(vae))
    print('  yolo bbox   :', ls(yolo))
    print('  sams        :', ls(sams))
    miss = []
    if not any('flux1-schnell' in f and f.endswith('.gguf') for f in ls(ggu)):
        miss.append('flux1-schnell-Q5_K_S.gguf (Cell 2)')
    if 'clip_l.safetensors' not in ls(clip):
        miss.append('clip_l.safetensors (Cell 2)')
    if not any('t5' in f for f in ls(clip)):
        miss.append('t5-v1_1-xxl-encoder-Q4_K_M.gguf (Cell 2)')
    if 'ae.safetensors' not in ls(vae):
        miss.append('ae.safetensors (FLUX VAE, Cell 2)')
    if not any('face' in x for x in ls(yolo)):
        miss.append('face_yolov8m.pt')
    if not any('hand' in x for x in ls(yolo)):
        miss.append('hand_yolov8s.pt')
    if not any('foot' in x for x in ls(yolo)):
        miss.append('foot_anime_yolo11m_v3.pt (YOLO chân anime Claquasse)')
    if not any(x.endswith('.pth') for x in ls(sams)):
        miss.append('sam_vit_b_01ec64.pth')
    if miss:
        log('⚠️ Thiếu: ' + ' | '.join(miss))
    else:
        log('✅ Đủ FLUX GGUF + VAE + CLIP (L+T5) + YOLO mặt/tay/chân + SAM')

if KILL_OLD:
    log('Dừng ComfyUI / cloudflared cũ...')
    os.system('pkill -f "python main.py" >/dev/null 2>&1 || true')
    os.system('pkill -f cloudflared >/dev/null 2>&1 || true')
    time.sleep(2)

need_tunnel = TUNNEL.startswith('cloudflared')
if need_tunnel and not shutil.which('cloudflared') and not os.path.exists('/usr/local/bin/cloudflared'):
    log('Cài cloudflared...')
    os.system('wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb')
    os.system('dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

cmd = [sys.executable, 'main.py',
       '--listen', str(LISTEN), '--port', str(int(PORT)),
       '--preview-method', PREVIEW,
       '--output-directory', OUTPUT_DIR,
       '--input-directory', INPUT_DIR,
       '--temp-directory', TEMP_DIR,
       '--cuda-device', str(int(CUDA_DEVICE))]
if CORS:
    cmd += ['--enable-cors-header', CORS_ORIGIN or '*']
vram_map = {'highvram':'--highvram','normalvram':'--normalvram','lowvram':'--lowvram','novram':'--novram','cpu':'--cpu'}
if VRAM in vram_map: cmd.append(vram_map[VRAM])
if RESERVE_VRAM_GB and RESERVE_VRAM_GB > 0:
    cmd += ['--reserve-vram', str(float(RESERVE_VRAM_GB))]
if CUDA_MALLOC: cmd.append('--cuda-malloc')
else: cmd.append('--disable-cuda-malloc')
if DISABLE_SMART_MEMORY: cmd.append('--disable-smart-memory')
if FORCE_PREC == 'fp16': cmd.append('--force-fp16')
elif FORCE_PREC == 'fp32': cmd.append('--force-fp32')
unet_map = {'fp16-unet':'--fp16-unet','bf16-unet':'--bf16-unet','fp8_e4m3fn-unet':'--fp8_e4m3fn-unet'}
if UNET_PREC in unet_map: cmd.append(unet_map[UNET_PREC])
vae_map = {'fp16-vae':'--fp16-vae','fp32-vae':'--fp32-vae','bf16-vae':'--bf16-vae','cpu-vae':'--cpu-vae'}
if VAE_PREC in vae_map: cmd.append(vae_map[VAE_PREC])
att_map = {'pytorch':'--use-pytorch-cross-attention','split':'--use-split-cross-attention',
           'quad':'--use-quad-cross-attention','sage':'--use-sage-attention','flash':'--use-flash-attention'}
if ATTENTION in att_map: cmd.append(att_map[ATTENTION])
if DISABLE_XFORMERS or ATTENTION != 'xformers (nếu có)':
    cmd.append('--disable-xformers')
if FAST: cmd.append('--fast')
if VERBOSE_LOG: cmd += ['--verbose', 'INFO']
if EXTRA_ARGS.strip(): cmd += EXTRA_ARGS.strip().split()

log('Lệnh ComfyUI:')
print(' ', ' '.join(cmd), flush=True)

log_path = '/content/comfyui.log'
comfy_log = open(log_path, 'w')
comfy = subprocess.Popen(cmd, stdout=comfy_log, stderr=subprocess.STDOUT, cwd=COMFY)
open('/content/comfy.pid', 'w').write(str(comfy.pid))
log(f'pid={comfy.pid} — đợi cổng {PORT} (tối đa {WAIT_SECONDS}s)...')

def port_open():
    try:
        with socket.create_connection(('127.0.0.1', int(PORT)), timeout=1):
            return True
    except OSError:
        return False

ok_port = False
for i in range(int(WAIT_SECONDS)):
    time.sleep(1)
    if comfy.poll() is not None:
        print('\n----- /content/comfyui.log -----')
        os.system('tail -60 /content/comfyui.log')
        raise RuntimeError('❌ ComfyUI chết khi khởi động. Đổi VRAM=lowvram hoặc tắt FAST rồi chạy lại.')
    if i in (15,30,60,90,120):
        log(f'  ... {i}s')
        if VERBOSE_LOG: os.system('tail -4 /content/comfyui.log')
    if port_open():
        ok_port = True; break
if not ok_port:
    os.system('tail -60 /content/comfyui.log')
    raise RuntimeError(f'❌ Quá {WAIT_SECONDS}s chưa mở cổng {PORT}')
log(f'✅ ComfyUI listen {LISTEN}:{PORT}')
try:
    with urllib.request.urlopen(f'http://127.0.0.1:{int(PORT)}/system_stats', timeout=10) as r:
        log(f'   /system_stats HTTP {r.status}')
except Exception as e:
    log(f'   /system_stats: {e}')

url = None
if need_tunnel:
    proto = 'http2' if 'http2' in TUNNEL else 'quic'
    cf_log_path = '/content/cloudflared.log'
    cf_log = open(cf_log_path, 'w')
    cf_cmd = ['cloudflared','tunnel','--url',f'http://127.0.0.1:{int(PORT)}',
              '--http-host-header',f'127.0.0.1:{int(PORT)}','--protocol',proto]
    log('Tunnel: ' + ' '.join(cf_cmd))
    cf = subprocess.Popen(cf_cmd, stdout=cf_log, stderr=subprocess.STDOUT)
    open('/content/cloudflared.pid','w').write(str(cf.pid))
    pat = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
    for i in range(90):
        time.sleep(1)
        txt = open(cf_log_path, errors='ignore').read()
        m = pat.search(txt)
        if m:
            url = m.group(0).rstrip('/'); break
        if cf.poll() is not None:
            print(open(cf_log_path, errors='ignore').read()[-2000:])
            log('⚠️ cloudflared chết — Cell 5'); break
    if url:
        open('/content/comfy_url.txt','w').write(url)
        try:
            with urllib.request.urlopen(url + '/system_stats', timeout=25) as r:
                log(f'Tunnel HTTP {r.status}')
        except Exception as e:
            log(f'Tunnel HTTP: {e} (403 thường do bấm link trong Colab — hãy DÁN tab mới)')
else:
    log('Không bật tunnel. ComfyUI chỉ local 127.0.0.1')

print()
print('='*64)
if url:
    print('🎨 COPY LINK, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI (đừng bấm trong Colab):\n')
    print('   ' + url)
elif need_tunnel:
    print('⚠️ Chưa có link Cloudflare → Cell 5')
else:
    print(f'Local: http://127.0.0.1:{int(PORT)}')
print('='*64)
print()
print('Trong ComfyUI:')
print('  1. Kéo file workflow_flux_schnell_gguf_toi_uu.json (hoặc Load từ /input)')
print('  2. Nhấn R (Refresh)  3. Queue Prompt')
print('  4. Model tự động nạp: UnetLoaderGGUF → flux1-schnell-Q5_K_S.gguf')
print('     DualCLIPLoaderGGUF → clip_l.safetensors + t5-v1_1-xxl-encoder-Q4_K_M.gguf (type=flux)')
print('     VAELoader → ae.safetensors')
print()
print('💡 ComfyUI vẫn chạy nền sau khi cell xong.')
print()
print('🖌  SỬA VÙNG LỖI (inpaint):')
print('   - Cách nhanh: chạy Cell 6 bên dưới (giao diện Gradio vẽ tay trên ảnh, dùng GGUF inpaint workflow).')
print('   - Trong ComfyUI: bấm phải ảnh → Open in MaskEditor → tô đen vùng lỗi → Save to node')
print('     rồi nối mask vào VAEEncodeForInpaint + KSampler(denoise=0.45, steps=8, cfg=1.0).')

# Thông báo model đang chạy
print()
print('='*64)
print('⚡ Model: FLUX.1-schnell GGUF Q5_K_S (chất lượng cao nhất dưới 15GB)')
print('   VRAM khuyến nghị: lowvram (bắt buộc trên T4 16GB), --fp16-vae, pytorch attention')
print('   Tham số: steps=4, CFG=1.0, sampler=euler, scheduler=simple, 1024×1024')
print('   Sau khi ảnh xong, FaceDetailer tự động fix mặt/tay/chân (tiled_encode/tiled_decode).')
print('   Chạy Cell 3C để copy workflow vào ComfyUI/input/.')
print('='*64)
print()
print('OOM / đỏ VRAM: chạy lại Cell 3 với VRAM=lowvram, VAE_PREC=cpu-vae, PREVIEW=none.')


In [ ]:
# ===== CELL 3B (TÙY CHỌN — không dùng với bản FLUX-only) =====
# Cell này giữ lại từ bản cũ (giao diện Gradio tiếng Việt cho WAI).
# Với bản chỉ FLUX GGUF, bạn KHÔNG cần chạy cell này — hãy dùng trực tiếp giao diện ComfyUI
# hoặc Cell 6 bên dưới (inpaint vẽ tay).
# Nếu vẫn muốn chạy, bỏ chú thích 2 dòng dưới:
# !pip install -q gradio websocket-client
# !wget -q -O /content/giaodien_tao_anh.py https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/arena/01a09a8b-t-i-li-u/giaodien_tao_anh.py && python /content/giaodien_tao_anh.py
print("ℹ️  Cell 3B bị tắt — bản này chỉ chạy FLUX GGUF. Dùng ComfyUI trực tiếp hoặc Cell 6 (inpaint).")


In [ ]:
# ===== CELL 3C: Tải workflow FLUX vào /content + ComfyUI/input (FIX branch) =====
import os, subprocess, shutil, requests
# Tự tìm branch có file realistic mới
BRANCHES_TO_TRY = ["arena/01a0d36a-t-i-li-u", "main", "arena/01a0cc76-t-i-li-u"]
BASE = None
BRANCH = "main"
for br in BRANCHES_TO_TRY:
    url_test = f"https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/{br}/workflow_flux_schnell_Q5_realistic.json"
    try:
        r = requests.head(url_test, timeout=5)
        if r.status_code == 200:
            BRANCH = br
            BASE = f"https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/{BRANCH}"
            print(f"✅ Tìm thấy branch có workflow realistic: {BRANCH}")
            break
    except Exception as e:
        continue

if BASE is None:
    BRANCH = "arena/01a0d36a-t-i-li-u"
    BASE = f"https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/{BRANCH}"

files = [
    ("QUY_TRINH_FLUX.md", "/content/"),
    ("QUY_TRINH_FLUX_REALISTIC.md", "/content/"),
    ("workflow_flux_schnell_gguf_toi_uu.json", "/content/"),
    ("workflow_flux_schnell_Q5_realistic.json", "/content/"),
    ("workflow_flux_schnell_Q5_realistic_hires.json", "/content/"),
    ("workflow_flux_schnell_toi_uu.json", "/content/"),
    ("workflow_flux_dev_toi_uu.json", "/content/"),
]

os.chdir("/content")
for fname, dest_dir in files:
    url = f"{BASE}/{fname}"
    dest = os.path.join(dest_dir, fname)
    print(f"⬇️  {fname} từ {BRANCH}")
    r = subprocess.run(["curl", "-L", "-o", dest, url])
    if r.returncode==0 and os.path.isfile(dest) and os.path.getsize(dest) > 100:
        print(f"  ✅ {fname} ({os.path.getsize(dest)/1024:.1f} KB)")
        if fname.endswith(".json"):
            try:
                os.makedirs("/content/ComfyUI/input", exist_ok=True)
                subprocess.run(["cp", dest, f"/content/ComfyUI/input/{fname}"])
            except:
                pass
    else:
        print(f"  ❌ Lỗi tải {fname}")

print("\n--- File trong /content ---")
subprocess.run("ls -lh /content/*.json /content/*.md 2>&1 | tail -n 40", shell=True)
print("\n--- ComfyUI/input ---")
subprocess.run("ls -lh /content/ComfyUI/input/*.json 2>&1 | tail -n 20", shell=True)
print("\n--- Checkpoints ---")
subprocess.run("ls -lh /content/ComfyUI/models/checkpoints/ 2>&1 | tail -n 20", shell=True)
print("\n--- UNET GGUF ---")
subprocess.run("ls -lh /content/ComfyUI/models/unet/ 2>&1 | tail -n 20", shell=True)
print("\n--- CLIP ---")
subprocess.run("ls -lh /content/ComfyUI/models/clip/ 2>&1 | tail -n 20", shell=True)
print("\n✅ Xong! Workflow realistic mới đã tải.")



In [ ]:
# ===== CELL 4: Kiểm tra =====
!curl -s -o /dev/null -w "A) ComfyUI: HTTP %{http_code}\n" --max-time 20 http://127.0.0.1:8188/system_stats
import re, os
txt = open('/content/cloudflared.log').read() if os.path.exists('/content/cloudflared.log') else ''
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
print('Link:', m.group(0) if m else '(chưa có)')
print('\nImpact:'); !ls /content/ComfyUI/custom_nodes | grep -i impact || echo THIEU
print('\nYOLO:'); !ls -lh /content/ComfyUI/models/ultralytics/bbox/ 2>/dev/null || echo THIEU
print('\nSAM:'); !ls -lh /content/ComfyUI/models/sams/ 2>/dev/null || echo THIEU
print('\nlog:'); !tail -30 /content/comfyui.log


In [ ]:
# ===== CELL 5: localtunnel dự phòng =====
!npm install -g localtunnel > /dev/null 2>&1
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('🔑 Tunnel Password:', ip)
!lt --port 8188


In [ ]:
# ===== CELL 6: 🎨 INPAINT VÙNG LỖI (vẽ tay bằng Gradio — chạy FLUX GGUF) - FIX 2026-09-24 =====
# Cách dùng:
#   1. Phải chạy xong Cell 3 (ComfyUI đang chạy nền ở http://127.0.0.1:8188)
#   2. Chạy cell này, đợi hiện link https://xxxx.gradio.live
#   3. Upload ảnh gốc (hoặc để trống → tự lấy ảnh mới nhất trong /content/ComfyUI/output/)
#   4. Dùng chuột TÔ ĐEN/TRẮNG lên vùng bị lỗi (tay, chân, mặt)
#   5. Mô tả phần bạn muốn vẽ lại (VD: "five fingers, natural hand, detailed skin")
#   6. Bấm "Sửa vùng tô" — chờ 15-25 giây trên T4

import os, io, time, json, uuid, shutil, random, sys, subprocess
import requests
from PIL import Image
import numpy as np

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio', 'websocket-client'], check=True)
    import gradio as gr

COMFY = "http://127.0.0.1:8188"

# Kiểm tra ComfyUI đang chạy chưa
def check_comfy():
    try:
        r = requests.get(f"{COMFY}/system_stats", timeout=5)
        return r.status_code == 200
    except Exception as e:
        return False

if not check_comfy():
    print("❌ ComfyUI chưa chạy! Chạy Cell 3 trước rồi mới chạy Cell 6.")
    print(f"   Thử curl: curl {COMFY}/system_stats")
    # vẫn tiếp tục để user thấy UI báo lỗi

def find_latest_output():
    out_dir = "/content/ComfyUI/output"
    if not os.path.isdir(out_dir): 
        return None
    files = [os.path.join(out_dir,f) for f in os.listdir(out_dir) if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))]
    if not files: 
        return None
    return max(files, key=os.path.getmtime)

def parse_gradio_mask(img_input):
    """
    Xử lý input từ Gradio Image Editor / Sketchpad / ImageMask - tương thích Gradio 3,4,5
    Trả về (background PIL RGB, mask PIL L)
    """
    bg = None
    mask = None

    # Gradio 5: ImageEditor trả về dict {'background': ..., 'layers': [...], 'composite': ...}
    if isinstance(img_input, dict):
        # case 1: {'image': PIL, 'mask': PIL} - Gradio 3/4 sketch
        if 'image' in img_input and 'mask' in img_input:
            bg_raw = img_input['image']
            mask_raw = img_input['mask']
            # bg_raw có thể là PIL hoặc dict hoặc filepath
            if isinstance(bg_raw, dict) and 'name' in bg_raw:
                bg = Image.open(bg_raw['name']).convert("RGB")
            elif isinstance(bg_raw, str) and os.path.isfile(bg_raw):
                bg = Image.open(bg_raw).convert("RGB")
            else:
                bg = bg_raw.convert("RGB") if hasattr(bg_raw, 'convert') else Image.fromarray(bg_raw).convert("RGB")
            
            if isinstance(mask_raw, dict) and 'name' in mask_raw:
                mask = Image.open(mask_raw['name']).convert("L")
            elif isinstance(mask_raw, str) and os.path.isfile(mask_raw):
                mask = Image.open(mask_raw).convert("L")
            else:
                # mask có thể là RGBA, lấy alpha hoặc convert
                if hasattr(mask_raw, 'mode'):
                    if mask_raw.mode == 'RGBA':
                        mask = mask_raw.split()[-1]
                    else:
                        mask = mask_raw.convert("L")
                else:
                    mask = Image.fromarray(mask_raw).convert("L")
        # case 2: ImageEditor Gradio 5
        elif 'background' in img_input:
            bg_raw = img_input['background']
            if isinstance(bg_raw, dict) and 'name' in bg_raw:
                bg = Image.open(bg_raw['name']).convert("RGB")
            elif isinstance(bg_raw, str) and os.path.isfile(bg_raw):
                bg = Image.open(bg_raw).convert("RGB")
            elif hasattr(bg_raw, 'convert'):
                bg = bg_raw.convert("RGB")
            else:
                bg = Image.fromarray(bg_raw).convert("RGB")
            
            # layers chứa mask vẽ
            layers = img_input.get('layers', [])
            composite = img_input.get('composite', None)
            if composite is not None:
                # composite là ảnh đã ghép, nhưng ta cần mask
                # Tạo mask từ layers: nếu có layers, lấy alpha của layer đầu
                if layers:
                    layer0 = layers[0]
                    if isinstance(layer0, dict) and 'name' in layer0:
                        layer_img = Image.open(layer0['name']).convert("RGBA")
                    elif isinstance(layer0, str) and os.path.isfile(layer0):
                        layer_img = Image.open(layer0).convert("RGBA")
                    elif hasattr(layer0, 'convert'):
                        layer_img = layer0.convert("RGBA")
                    else:
                        layer_img = Image.fromarray(layer0).convert("RGBA")
                    # mask là alpha channel
                    mask = layer_img.split()[-1]
                else:
                    # không có layer, thử dùng composite vs background diff
                    if isinstance(composite, dict) and 'name' in composite:
                        comp_img = Image.open(composite['name']).convert("RGB")
                    elif isinstance(composite, str) and os.path.isfile(composite):
                        comp_img = Image.open(composite['name']).convert("RGB")
                    elif hasattr(composite, 'convert'):
                        comp_img = composite.convert("RGB")
                    else:
                        comp_img = Image.fromarray(composite).convert("RGB")
                    # fallback: tạo mask trắng toàn bộ (sẽ inpaint cả ảnh - cảnh báo)
                    mask = Image.new("L", bg.size, 255)
            else:
                mask = Image.new("L", bg.size, 255)
        else:
            # dict lạ, thử lấy image key
            print(f"⚠️ Dict keys không nhận diện: {list(img_input.keys())}")
            return None, None
    elif isinstance(img_input, (list, tuple)) and len(img_input) == 2:
        # Gradio 4 ImageMask trả về tuple (image, mask)
        bg_raw, mask_raw = img_input
        bg = bg_raw.convert("RGB") if hasattr(bg_raw, 'convert') else Image.fromarray(bg_raw).convert("RGB")
        mask = mask_raw.convert("L") if hasattr(mask_raw, 'convert') else Image.fromarray(mask_raw).convert("L")
    elif hasattr(img_input, 'convert'):
        # Chỉ có ảnh nền, không có mask -> báo lỗi
        bg = img_input.convert("RGB")
        mask = None
    else:
        print(f"⚠️ Kiểu input không hỗ trợ: {type(img_input)}")
        return None, None

    return bg, mask

def run_inpaint(img_input, prompt, negative, denoise, grow_mask, seed, cfg, steps):
    if img_input is None:
        latest = find_latest_output()
        if latest is None:
            return None, "❌ Chưa upload ảnh và cũng chưa có ảnh nào trong /content/ComfyUI/output/", None
        bg = Image.open(latest).convert("RGB")
        mask = Image.new("L", bg.size, 0)
        print(f"📂 Tự lấy ảnh mới nhất: {latest}")
    else:
        bg, mask = parse_gradio_mask(img_input)
        if bg is None:
            return None, f"❌ Không đọc được ảnh input: {type(img_input)} - Thử upload lại", None
        if mask is None:
            return None, "❌ Chưa vẽ mask! Dùng brush tô lên vùng cần sửa (màu trắng là vùng sẽ inpaint)", None

    # Đảm bảo mask có kích thước bằng bg
    if mask.size != bg.size:
        mask = mask.resize(bg.size)

    # Chuyển mask về trắng = vùng cần inpaint
    # Gradio mask thường đen = giữ, trắng = vẽ. Nhưng user tô đen trong hướng dẫn cũ -> ta chuẩn hóa:
    # Nếu mask trung bình > 128 nghĩa là user tô trắng nhiều -> giữ nguyên. Nếu < 20 nghĩa là mask rỗng
    mask_np = np.array(mask)
    white_ratio = np.mean(mask_np > 128)
    if white_ratio < 0.01:
        return None, "❌ Mask rỗng - bạn chưa tô gì cả! Dùng brush tô trắng lên vùng lỗi", None

    # Nếu mask là đen trên nền trắng (ngược), đảo lại? Ta giữ quy ước: trắng = inpaint
    # Kiểm tra nếu mask toàn đen với vài nét trắng nhỏ thì ok
    # Nếu mask toàn trắng -> user chưa xóa gì, cảnh báo
    if white_ratio > 0.9:
        # Có thể user tô toàn bộ -> vẫn cho chạy nhưng cảnh báo
        print("⚠️ Mask gần như toàn trắng - sẽ inpaint cả ảnh!")

    # Lưu tạm
    rid = uuid.uuid4().hex[:10]
    img_path = f"/tmp/inp_{rid}.png"
    msk_path = f"/tmp/msk_{rid}.png"
    bg.save(img_path)
    # Đảm bảo mask là RGB để ImageToMask channel red hoạt động
    mask_rgb = Image.new("RGB", bg.size, (0,0,0))
    mask_rgb.paste(Image.fromarray(mask_np), mask=Image.fromarray(mask_np))
    # Đơn giản hơn: lưu mask L rồi convert sang RGB khi upload
    mask_for_save = mask.convert("RGB")
    mask_for_save.save(msk_path)

    def upload(p):
        with open(p, 'rb') as f:
            r = requests.post(f"{COMFY}/upload/image",
                              files={"image": f},
                              data={"overwrite":"true","type":"input","subfolder":""},
                              timeout=30)
        r.raise_for_status()
        return r.json()["name"]
    try:
        img_name = upload(img_path)
        msk_name = upload(msk_path)
        print(f"✅ Upload: {img_name}, {msk_name}")
    except Exception as e:
        return None, f"❌ Upload ảnh lên ComfyUI thất bại: {e}\nKiểm tra Cell 3 đang chạy: curl {COMFY}/system_stats", None

    neg_text = negative.strip() if negative.strip() else ""
    seed_v = int(seed) if int(seed) >= 0 else random.randint(0, 2**31-1)
    ctrl = "fixed" if int(seed) >= 0 else "randomize"

    # FLUX GGUF inpaint workflow - tối ưu người thật
    workflow = {
      "1":  {"class_type":"UnetLoaderGGUF",
             "inputs":{"unet_name":"flux1-schnell-Q5_K_S.gguf"}},
      "2":  {"class_type":"DualCLIPLoaderGGUF",
             "inputs":{"clip_name1":"clip_l.safetensors",
                       "clip_name2":"t5-v1_1-xxl-encoder-Q4_K_M.gguf",
                       "type":"flux"}},
      "3":  {"class_type":"VAELoader",
             "inputs":{"vae_name":"ae.safetensors"}},
      "4":  {"class_type":"CLIPTextEncode",
             "inputs":{"text": prompt, "clip":["2",0]}},
      "5":  {"class_type":"CLIPTextEncode",
             "inputs":{"text": neg_text, "clip":["2",0]}},
      "6":  {"class_type":"LoadImage","inputs":{"image": img_name}},
      "7":  {"class_type":"LoadImage","inputs":{"image": msk_name}},
      "8":  {"class_type":"ImageToMask","inputs":{"image":["7",0],"channel":"red"}},
      "9":  {"class_type":"VAEEncodeForInpaint",
             "inputs":{"pixels":["6",0],"mask":["8",0],"vae":["3",0],
                       "grow_mask_by": int(grow_mask)}},
      "10": {"class_type":"KSampler",
             "inputs":{"seed": seed_v,
                       "control_after_generate": ctrl,
                       "steps": int(steps),
                       "cfg": float(cfg),
                       "sampler_name":"euler",
                       "scheduler":"simple",
                       "denoise": float(denoise),
                       "model":["1",0],
                       "positive":["4",0],
                       "negative":["5",0],
                       "latent_image":["9",0]}},
      "11": {"class_type":"VAEDecode",
             "inputs":{"samples":["10",0],"vae":["3",0]}},
      "12": {"class_type":"SaveImage",
             "inputs":{"filename_prefix":f"Inpaint_GGUF_{rid}","images":["11",0]}}
    }

    try:
        r = requests.post(f"{COMFY}/prompt", json={"prompt": workflow}, timeout=15)
    except Exception as e:
        return None, f"❌ Không kết nối được ComfyUI: {e}", None
        
    if r.status_code != 200:
        return None, f"❌ Lỗi gửi prompt: HTTP {r.status_code} — {r.text[:500]}", None
    pid = r.json()["prompt_id"]
    print(f"📤 Đã gửi prompt_id={pid}, chờ ComfyUI...")

    out_path = None
    for i in range(180):
        time.sleep(1)
        try:
            h = requests.get(f"{COMFY}/history/{pid}", timeout=10).json()
        except Exception:
            continue
        if pid in h:
            outputs = h[pid].get("outputs",{})
            for nid, node_out in outputs.items():
                if "images" in node_out:
                    im = node_out["images"][0]
                    sub = im.get("subfolder","")
                    fn = im["filename"]
                    base = "/content/ComfyUI/output" if im["type"]=="output" else "/content/ComfyUI/input"
                    out_path = os.path.join(base, sub, fn) if sub else os.path.join(base, fn)
                    break
            break
        if i % 10 == 0:
            print(f"  ... chờ {i}s (prompt {pid})")

    if not out_path or not os.path.isfile(out_path):
        return None, f"❌ Hết giờ chờ hoặc ComfyUI lỗi - kiểm tra log: tail /content/comfyui.log\nPrompt_id: {pid}", None

    final = f"/tmp/inp_out_{rid}.png"
    shutil.copy(out_path, final)
    msg = (f"✅ Xong — denoise={denoise}, grow={int(grow_mask)}px, steps={int(steps)}, cfg={cfg}, seed={seed_v}\n"
           f"File gốc: {os.path.basename(out_path)}")
    return Image.open(final), msg, final

default_img_path = find_latest_output()
default_img_pil = None
if default_img_path:
    try:
        default_img_pil = Image.open(default_img_path).convert("RGB")
        print(f"📂 Ảnh mặc định: {default_img_path}")
    except:
        default_img_pil = None

# Tạo UI tương thích Gradio 3,4,5
def create_ui():
    # Thử Gradio 5 ImageEditor
    has_image_editor = hasattr(gr, 'ImageEditor')
    has_sketchpad = hasattr(gr, 'Sketchpad')
    has_image_mask = hasattr(gr, 'ImageMask')
    
    print(f"Gradio version: {gr.__version__}, has ImageEditor={has_image_editor}, Sketchpad={has_sketchpad}, ImageMask={has_image_mask}")

    with gr.Blocks(title="🎨 Inpaint FLUX GGUF - Fix") as demo:
        gr.Markdown("## 🎨 Inpaint vùng lỗi — FLUX.1-schnell GGUF Q5_K_S (FIX 2026-09-24)\n"
                    "1. Upload ảnh (để trống sẽ tự lấy ảnh mới nhất từ output)\n"
                    "2. **Tô trắng** vùng bị lỗi bằng brush (tay, mặt, chân)\n"
                    "3. Mô tả phần muốn vẽ lại → bấm **Sửa vùng tô**\n"
                    "   ⏱ ~15-25s/ảnh trên T4, model ~12GB tổng\n"
                    "   💡 Prompt người thật: `detailed hand, five fingers, natural skin texture, photorealistic`")

        with gr.Row():
            with gr.Column():
                # Chọn component phù hợp
                if has_image_editor:
                    # Gradio 5
                    img_in = gr.ImageEditor(
                        value={"background": default_img_pil, "layers": [], "composite": default_img_pil} if default_img_pil else None,
                        label="Ảnh gốc — VẼ TRẮNG lên vùng cần sửa (dùng brush)",
                        type="pil",
                        height=640,
                        brush=gr.Brush(colors=["#FFFFFF"], color_mode="fixed", default_size=40)
                    )
                elif has_sketchpad:
                    img_in = gr.Sketchpad(
                        value=default_img_pil,
                        label="Ảnh gốc — TÔ TRẮNG vùng cần sửa",
                        type="pil",
                        height=640,
                        brush_radius=40
                    )
                elif has_image_mask:
                    img_in = gr.ImageMask(
                        value=default_img_pil,
                        label="Ảnh gốc — TÔ TRẮNG vùng cần sửa",
                        type="pil",
                        height=640
                    )
                else:
                    # Fallback Gradio 3
                    try:
                        img_in = gr.Image(
                            value=default_img_pil,
                            label="Ảnh gốc — TÔ ĐEN vùng cần sửa",
                            tool="sketch",
                            type="pil",
                            height=640
                        )
                    except TypeError:
                        # Gradio 4+ không còn tool
                        img_in = gr.Image(
                            value=default_img_pil,
                            label="Ảnh gốc — Upload rồi dùng tab Mask bên dưới nếu không vẽ được",
                            type="pil",
                            height=640
                        )

                prompt = gr.Textbox(label="✍️ Mô tả phần muốn vẽ lại (tiếng Anh, tự nhiên)",
                                    value="detailed hand, five fingers, natural fingernails, natural skin texture, photorealistic, sharp focus, detailed skin",
                                    lines=2,
                                    placeholder="VD: photorealistic face, natural skin texture, sharp eyes...")
                neg = gr.Textbox(label="🚫 Negative (FLUX có thể để trống)",
                                 value="cartoon, anime, blurry, deformed, plastic skin",
                                 lines=1)
                with gr.Row():
                    denoise = gr.Slider(0.2, 0.85, value=0.5, step=0.05, label="Denoise (0.4=giữ, 0.7=vẽ lại)")
                    grow = gr.Slider(0, 32, value=12, step=2, label="Grow mask px")
                    seed = gr.Number(value=-1, label="Seed (-1=random)")
                with gr.Row():
                    cfg = gr.Slider(1.0, 5.0, value=1.0, step=0.5, label="CFG (FLUX=1.0)")
                    steps = gr.Slider(4, 20, value=8, step=1, label="Steps (8)")
                btn = gr.Button("🖌 Sửa vùng tô", variant="primary", size="lg")
                
                gr.Markdown("**Mẹo người thật:**\n"
                            "- Mặt: `photorealistic face, natural skin texture, visible pores, sharp eyes`\n"
                            "- Tay: `five fingers, perfect hands, detailed fingernails, realistic skin`\n"
                            "- Denoise mặt 0.4-0.5, tay 0.5-0.6")

            with gr.Column():
                img_out = gr.Image(label="Kết quả", height=640, type="pil")
                msg = gr.Markdown(value="Sẵn sàng. Upload ảnh + tô mask + bấm Sửa.")
                dl = gr.File(label="⬇️ Tải về", interactive=False)

        def do_run(img_dict, p, n, d, g, s, c, st):
            from PIL import ImageFile
            ImageFile.LOAD_TRUNCATED_IMAGES = True
            out, m, path = run_inpaint(img_dict, p, n, d, g, s, c, st)
            if out is None:
                return None, m, None
            out_path = f"/tmp/inpaint_final_{uuid.uuid4().hex[:8]}.png"
            out.save(out_path)
            return out, m, out_path

        btn.click(do_run, inputs=[img_in, prompt, neg, denoise, grow, seed, cfg, steps],
                  outputs=[img_out, msg, dl])

    return demo

demo = create_ui()
print("Đợi link https://xxxx.gradio.live — bấm STOP khi muốn tắt.\n")
print(f"ComfyUI check: {'✅ OK' if check_comfy() else '❌ Chưa chạy Cell 3'}")
demo.queue().launch(share=True, server_name="0.0.0.0", server_port=7860)



## 🔧 Đã làm gì (bản FLUX.1-schnell GGUF Q5_K_S — only FLUX, <15GB)

- **Mặc định chỉ có 1 model**: `FLUX.1-schnell Q5_K_S` (UNET) + `t5-v1_1-xxl-Q4_K_M` (T5-XXL) + `ae.safetensors` (VAE) + `clip_l.safetensors` (CLIP-L) — tổng ~12.3GB.
- Tự động cài **ComfyUI-GGUF** (city96) trong Cell 1B.
- Symlink thêm `models/vae`, `models/clip`, `models/unet` vào Drive để model GGUF được nhận.
- Workflow chính `workflow_flux_schnell_gguf_toi_uu.json` (18 node, đã tối ưu):
  - `UnetLoaderGGUF → flux1-schnell-Q5_K_S.gguf`
  - `DualCLIPLoaderGGUF → clip_l + t5-xxl-Q4_K_M (type=flux)`
  - `VAELoader → ae.safetensors`
  - KSampler euler/simple/4 steps/CFG 1.0 — chuẩn schnell
  - 3 FaceDetailer: mặt (SAM+tiled), tay/chân (feather=24, no-SAM, tiled)
- **Cell 6 Gradio inpaint** giờ chạy được GGUF (dùng UnetLoaderGGUF + DualCLIPLoaderGGUF + VAELoader + VAEEncodeForInpaint), steps=8/CFG=1.0/denoise=0.5 khuyến nghị.
- Không còn WAI/FP8/dev, mọi thứ tập trung vào **phiên bản FLUX tốt nhất dưới 15GB**.

### Kích thước Q5_K_S vs các bản khác
| Quant | UNET | UNET+T5+CLIP+VAE |
|---|---|---|
| Q2_K | 4.0 GB | ~7.5 GB |
| Q3_K_S | 5.2 GB | ~8.7 GB |
| Q4_K_S | 6.8 GB | ~10.3 GB |
| **Q5_K_S** | **8.3 GB** | **~11.8 GB** ← chọn |
| Q6_K | 9.8 GB | ~13.3 GB (gần 15GB quá) |
| Q8_0 | 12.7 GB | ~16 GB (vượt 15GB) |
